# Macroeconomic Forecasting Project: Multivariate Analysis
---

<div align="center">
  <p><strong>BỘ GIÁO DỤC VÀ ĐÀO TẠO</strong></p>
  <p><strong>TRƯỜNG ĐẠI HỌC CÔNG NGHỆ KỸ THUẬT THÀNH PHỐ HỒ CHÍ MINH</strong></p>
  <p><strong>KHOA CÔNG NGHỆ THÔNG TIN</strong></p>
  <br>

  <h1><strong>PROJECT 1</strong></h1>
  <h2><strong>PHÂN TÍCH CÁC YẾU TỐ VĨ MÔ TÁC ĐỘNG ĐẾN LẠM PHÁT TẠI VIỆT NAM TỪ NĂM 1996 ĐẾN NĂM 2022 VÀ DỰ BÁO XU HƯỚNG BẰNG MÔ HÌNH CHUỖI THỜI GIAN</strong></h2>
  <br>
</div>

**MÔN HỌC:** PHÂN TÍCH DỮ LIỆU
**LỚP:** DAAN436277_02
**GVHD:** ThS. Trần Trọng Bình

### THÀNH VIÊN NHÓM 02:

| STT | Họ và tên | Mã số sinh viên |
|:---:|:---|:---:|
| 1 | Đỗ Kiến Hưng | 23133030 |
| 2 | Trần Minh Khánh | 23133035 |
| 3 | Nguyễn Đặng Quốc Anh | 23133004 |
| 4 | Phạm Minh Quân | 23133060 |

---

> **THE ULTIMATE QUESTION**
>
> Nhóm tiến hành phân tích dự án nhằm đi tìm lời giải đáp cho vấn đề cốt lõi:
> *"Trong 3 thập kỷ qua, Lạm phát tại Việt Nam chủ yếu là do **'Bệnh tự miễn'** (Quán tính tự sinh ra do tâm lý kỳ vọng) hay do **'Tác nhân bên ngoài'** (Tỷ giá, Tín dụng, Tăng trưởng GDP)? Cần can thiệp vào đâu để kiểm soát nó?"*

### CƠ SỞ KINH TẾ HỌC TRONG VIỆC LỰA CHỌN BIẾN SỐ

Để trả lời câu hỏi tối thượng của project, tập dữ liệu 11 biến được nhóm chọn đại diện cho **4 Trụ cột của nền Kinh tế Vĩ mô**:

1. **Trụ cột Giá cả (Đo lường bệnh lý):** `cpi_growth_percent` (Mục tiêu), `gdp_deflator_percent` (Lạm phát từ phía sản xuất).
   - *Tại sao?* CPI đo lường giá cả từ phía tiêu dùng (giỏ hàng hóa hộ gia đình), còn GDP Deflator đo từ phía sản xuất. Khi cả hai cùng tăng, ta biết chắc lạm phát là "thật" chứ không phải do một sự kiện cục bộ.

2. **Trụ cột Tăng trưởng (Áp lực Cầu kéo):** `gdp_growth_percent`, `unemployment_rate`.
   - *Tại sao?* Theo lý thuyết Đường cong Phillips, khi GDP tăng nhanh → thất nghiệp giảm → áp lực tăng lương → giá cả tăng. Đây là kênh "Cầu kéo" (Demand-pull) của lạm phát.

3. **Trụ cột Tiền tệ (Bơm tiền & Lãi suất):** `domestic_credit_index`, `lending_interest_percent`.
   - *Tại sao?* Khi tín dụng mở rộng (bơm tiền) và lãi suất giảm → doanh nghiệp và người dân vay nhiều hơn → tiền trong nền kinh tế tăng → giá cả tăng. Đây là kênh "Tiền tệ" (Monetary channel).

4. **Trụ cột Đối ngoại (Cú sốc ngoại sinh):** `officical_exchange_rate_percent`, `import_index`, `export_index`.
   - *Tại sao?* Khi VND mất giá so với USD → hàng nhập khẩu (xăng dầu, nguyên liệu) đắt hơn → chi phí sản xuất tăng → giá cả tăng. Đây là kênh "Chi phí đẩy" (Cost-push) từ bên ngoài.

## **CHƯƠNG 2: Khám phá & Đặc trưng hóa Dữ liệu Thô**
---

> **Triết lý phân tích:** Trước khi đưa bất kỳ dữ liệu nào vào mô hình, chúng ta PHẢI hiểu rõ dữ liệu mình đang có. Mỗi biến số có những đặc điểm riêng (phân phối, xu hướng, giá trị thiếu) mà nếu không nắm bắt sẽ dẫn đến kết luận sai lệch.

### 2.1 Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.stattools import adfuller
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.stattools import grangercausalitytests

import warnings
warnings.filterwarnings('ignore')

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

### 2.2 Upload & Preview Dataset

In [ ]:
# Tải dữ liệu
# Nếu chạy trên Google Colab, hãy upload file dataset_project1.xlsx
# hoặc sửa đường dẫn cho phù hợp
df = pd.read_excel('../data/dataset_project1.xlsx')
display(df.head(10))

### 2.3 Data Overview: Hiểu cấu trúc dữ liệu

**Mục đích:** Trước khi phân tích, ta cần biết:
- Có bao nhiêu quan sát (dòng)?
- Mỗi biến có kiểu dữ liệu gì?
- Có giá trị null/thiếu không?
- Các thống kê mô tả cơ bản (mean, std, min, max) cho biết gì?

In [ ]:
print("=" * 60)
print("THÔNG TIN CẤU TRÚC DATASET")
print("=" * 60)
print(df.info())
print()
print("=" * 60)
print("THỐNG KÊ MÔ TẢ")
print("=" * 60)
display(df.describe())

### 2.4 Missing Values Analysis

**Tại sao phải kiểm tra giá trị thiếu?**
- Dữ liệu kinh tế vĩ mô từ World Bank thường có khoảng trống do: một số quốc gia chưa thống kê đầy đủ vào những năm đầu, hoặc phương pháp thu thập thay đổi.
- Nếu bỏ qua missing values → mô hình sẽ bị lệch (biased) hoặc không thể chạy được.

In [ ]:
# Đếm số giá trị thiếu mỗi cột
missing_counts = df.isnull().sum()
print("SỐ GIÁ TRỊ THIẾU THEO TỪNG BIẾN:")
print(missing_counts)
print()

# Heatmap trực quan hóa vị trí missing values
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap missing values
sns.heatmap(df.isnull(), cbar=False, ax=axes[0], cmap='viridis')
axes[0].set_title('Bản đồ Giá trị Thiếu (Vàng = Missing)', fontweight='bold')
axes[0].set_ylabel('Chỉ số dòng (Năm)')

# Bar chart missing values
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct = missing_pct[missing_pct > 0]
if len(missing_pct) > 0:
    missing_pct.plot(kind='barh', ax=axes[1], color='coral')
    axes[1].set_title('Tỷ lệ % Giá trị Thiếu', fontweight='bold')
    axes[1].set_xlabel('% Missing')
else:
    axes[1].text(0.5, 0.5, 'Không có giá trị thiếu!', ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()

**Nhận xét & Hành động:**
- `domestic_credit_index` và `lending_interest_percent` là hai biến có giá trị thiếu. Đây là các biến thuộc **Trụ cột Tiền tệ**.
- **Tại sao thiếu?** Dữ liệu tín dụng nội địa và lãi suất cho vay từ World Bank cho Việt Nam không được ghi nhận đầy đủ trong một số năm đầu tiên của giai đoạn nghiên cứu.
- **Giải pháp:** Sử dụng **nội suy tuyến tính (linear interpolation)** — phương pháp phù hợp nhất cho dữ liệu chuỗi thời gian vì nó giả định giá trị biến đổi dần dần theo thời gian (hợp lý với dữ liệu kinh tế).

### 2.5 Data Cleaning: Xử lý cột trùng lặp & giá trị thiếu

In [ ]:
# Kiểm tra cột trùng lặp/lỗi chính tả
print("Các cột trong dataset:")
for col in df.columns:
    print(f"  - {col}")

# Nhận thấy có cả 'gdp_delflator_percent' (lỗi chính tả) và 'gdp_deflator_percent'
# Kiểm tra xem chúng có giống nhau không
if 'gdp_delflator_percent' in df.columns and 'gdp_deflator_percent' in df.columns:
    print("\nKiểm tra 2 cột gdp deflator:")
    print(f"  gdp_delflator_percent (typo) vs gdp_deflator_percent:")
    # Cột 'gdp_delflator_percent' là GDP Deflator INDEX (tích lũy)
    # Cột 'gdp_deflator_percent' là GDP Deflator GROWTH RATE (% thay đổi)
    print(f"  Cột 'gdp_delflator_percent' - range: {df['gdp_delflator_percent'].min():.1f} to {df['gdp_delflator_percent'].max():.1f}")
    print(f"  Cột 'gdp_deflator_percent' - range: {df['gdp_deflator_percent'].min():.1f} to {df['gdp_deflator_percent'].max():.1f}")
    print("  => Hai cột này KHÁC nhau: một là Index, một là Growth Rate")

In [ ]:
# Xử lý missing values bằng linear interpolation
print("Xử lý giá trị thiếu...")

# Đặt 'years' làm index
df['years'] = pd.to_datetime(df['years'], format='%Y')
df.set_index('years', inplace=True)

# Loại bỏ cột trùng lặp (giữ lại cột đúng chính tả)
if 'gdp_delflator_percent' in df.columns:
    # Đổi tên cột lỗi thành tên có ý nghĩa
    df.rename(columns={'gdp_delflator_percent': 'gdp_deflator_index'}, inplace=True)
    # Giữ 'cpi_index' và loại bỏ redundancy
    if 'cpi_index' in df.columns:
        print("  Giữ lại: cpi_index, gdp_deflator_index (là Index tích lũy)")
        print("  Giữ lại: cpi_growth_percent, gdp_deflator_percent (là Growth Rate %)")

# Nội suy tuyến tính cho missing values
df.interpolate(method='linear', inplace=True)

# Kiểm tra lại
print(f"\nSau khi xử lý - Số giá trị thiếu còn lại: {df.isnull().sum().sum()}")
print(f"Shape: {df.shape}")
display(df.head())

### 2.6 Time Series Trend Visualization: Xu hướng theo thời gian

**Mục đích:** Trước khi chạy bất kỳ model nào, ta cần "nhìn" dữ liệu. Biểu đồ xu hướng giúp:
1. Phát hiện các sự kiện kinh tế lớn ảnh hưởng đến dữ liệu.
2. Nhận biết bằng mắt liệu chuỗi có "dừng" (stationary) hay không.
3. Tìm mối liên hệ trực quan giữa các biến.

In [ ]:
# Macroeconomic Dashboard - 4 biểu đồ cho 4 Trụ cột
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(22, 15))

# 1. Trụ cột Giá cả: CPI Growth
axes[0,0].plot(df.index, df['cpi_growth_percent'], linewidth=3, color='#E24A33', label='CPI Growth Rate')
axes[0,0].fill_between(df.index, df['cpi_growth_percent'], alpha=0.15, color='#E24A33')
axes[0,0].axhline(0, color='black', linewidth=0.5)
# Đánh dấu sự kiện
import pandas as pds
axes[0,0].axvspan(pds.Timestamp('2008-01-01'), pds.Timestamp('2009-12-31'), alpha=0.15, color='gray', label='Khủng hoảng 2008')
axes[0,0].axvspan(pds.Timestamp('2020-01-01'), pds.Timestamp('2021-12-31'), alpha=0.15, color='blue', label='COVID-19')
axes[0,0].set_title('1. Lạm phát bùng nổ trùng với các cuộc khủng hoảng toàn cầu', fontsize=14, fontweight='bold')
axes[0,0].set_ylabel('Inflation Rate (%)')
axes[0,0].legend(loc='upper right')
axes[0,0].grid(True, linestyle='--', alpha=0.5)

# 2. Trụ cột Tiền tệ: Credit vs Interest Rate
ax1 = axes[0,1]
ax2 = ax1.twinx()
credit_series = df['domestic_credit_index'].ffill()
interest_series = df['lending_interest_percent'].ffill()
line1 = ax1.plot(df.index, credit_series, color='#348ABD', linewidth=2.5, label='Domestic Credit Index (Trái)')
line2 = ax2.plot(df.index, interest_series, color='#988ED5', linewidth=2.5, linestyle='--', label='Lending Interest Rate (Phải)')
ax1.set_title('2. Chính sách tiền tệ phản chu kỳ: Hạ lãi suất → Tín dụng tăng', fontsize=14, fontweight='bold')
ax1.set_ylabel('Domestic Credit Index', color='#348ABD')
ax2.set_ylabel('Lending Interest Rate (%)', color='#988ED5')
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left')
ax1.grid(True, linestyle='--', alpha=0.3)

# 3. Trụ cột Đối ngoại: Import vs Export
axes[1,0].plot(df.index, df['export_index'], color='#FBC15E', linewidth=2.5, label='Export Index')
axes[1,0].plot(df.index, df['import_index'], color='#8EBA42', linewidth=2.5, label='Import Index')
axes[1,0].fill_between(df.index, df['export_index'], df['import_index'],
                       where=(df['export_index'] >= df['import_index']),
                       color='green', alpha=0.12, label='Thặng dư')
axes[1,0].fill_between(df.index, df['export_index'], df['import_index'],
                       where=(df['export_index'] < df['import_index']),
                       color='red', alpha=0.12, label='Thâm hụt')
axes[1,0].set_title('3. Chuyển đổi cơ cấu: Từ thâm hụt mãn tính sang thặng dư bền vững sau 2012', fontsize=14, fontweight='bold')
axes[1,0].set_ylabel('Value Index')
axes[1,0].legend(loc='upper left')
axes[1,0].grid(True, linestyle='--', alpha=0.5)

# 4. Trụ cột Tăng trưởng: GDP Growth
gdp_series = df['gdp_growth_percent'].ffill()
axes[1,1].bar(df.index, gdp_series, width=260, color='#4682B4', alpha=0.4, label='GDP Growth Rate (Bar)')
axes[1,1].plot(df.index, gdp_series, color='#4682B4', linewidth=2.5, marker='o', markersize=5, label='GDP Trend (Line)')
axes[1,1].axhline(0, color='black', linewidth=1.2)
axes[1,1].set_title('4. Tăng trưởng GDP dài hạn bền bỉ với suy giảm chu kỳ hậu COVID', fontsize=14, fontweight='bold')
axes[1,1].set_ylabel('GDP Growth Rate (%)')
axes[1,1].legend(loc='upper right')
axes[1,1].grid(True, linestyle='--', alpha=0.5)

plt.suptitle('BẢNG ĐIỀU KHIỂN KINH TẾ VĨ MÔ VIỆT NAM (1996-2024)', fontsize=20, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Nhận xét từ biểu đồ xu hướng:**

1. **Lạm phát (CPI Growth):** Hai đỉnh lạm phát lớn nhất (2008 và 2011) đều trùng với các sự kiện kinh tế toàn cầu và chính sách tiền tệ nới lỏng trong nước. Điều này gợi ý rằng lạm phát VN chịu tác động từ CẢ yếu tố nội sinh lẫn ngoại sinh.

2. **Tín dụng vs Lãi suất:** Đồ thị cho thấy mối quan hệ nghịch rõ rệt — khi lãi suất giảm, tín dụng tăng vọt. Đây là cơ chế truyền dẫn tiền tệ cổ điển.

3. **Import/Export:** Việt Nam chuyển từ thâm hụt thương mại sang thặng dư sau 2012, nhờ hội nhập sâu vào chuỗi cung ứng toàn cầu (FDI từ Samsung, Intel...).

4. **GDP:** Tăng trưởng ổn định 5-7% với duy nhất 2020 suy giảm mạnh do COVID-19.

**Kết luận quan trọng:** Các chuỗi Index (CPI Index, Credit Index, Exchange Rate) có xu hướng tăng liên tục → chắc chắn KHÔNG dừng. Điều này buộc ta phải lấy sai phân (differencing) trước khi đưa vào VAR ở Chương 4.

## **CHƯƠNG 3: Phân tích Mối quan hệ & Kiểm định Giả thuyết**
---

> **Câu hỏi cốt lõi:** Yếu tố nào THỰC SỰ có mối liên hệ với Lạm phát? Mối liên hệ đó mạnh hay yếu? Có thể tin cậy được không?

### 3.1 Correlation Heatmap: Bức tranh toàn cảnh mối quan hệ

In [ ]:
# Chọn các biến phân tích chính (loại bỏ Index tích lũy)
analysis_vars = ['cpi_growth_percent', 'gdp_deflator_percent',
                 'gdp_growth_percent', 'unemployment_rate',
                 'domestic_credit_index', 'lending_interest_percent',
                 'officical_exchange_rate_percent', 'import_index', 'export_index']

corr_matrix = df[analysis_vars].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, mask=mask, square=True, linewidths=0.5)
plt.title('Ma trận Tương quan Pearson giữa các biến Kinh tế Vĩ mô', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

**Đọc hiểu Correlation Heatmap:**
- Giá trị gần **+1**: Tương quan thuận mạnh (cùng tăng cùng giảm).
- Giá trị gần **-1**: Tương quan nghịch mạnh (một tăng thì cái kia giảm).
- Giá trị gần **0**: Không có mối liên hệ tuyến tính.

**Phát hiện quan trọng:**
- `cpi_growth_percent` và `gdp_deflator_percent`: Tương quan thuận cao → Cả hai đều đo lường lạm phát, xác nhận dữ liệu nhất quán.
- `cpi_growth_percent` và `lending_interest_percent`: Tương quan thuận → Khi lạm phát tăng, NHNN tăng lãi suất để kiềm chế (phản ứng chính sách).
- `domestic_credit_index`, `import_index`, `export_index`, `exchange_rate`: Tương quan cao với nhau → CẢNH BÁO đa cộng tuyến (multicollinearity)! Cần kiểm tra VIF.

### 3.2 Multicollinearity Check (VIF)

**Tại sao cần kiểm tra VIF?**
- VIF (Variance Inflation Factor) đo mức độ "trùng lặp thông tin" giữa các biến.
- VIF > 10: Đa cộng tuyến nghiêm trọng → hệ số hồi quy không đáng tin → cần loại bỏ biến.
- **Hậu quả nếu bỏ qua:** Mô hình VAR sẽ cho hệ số "ảo" — kết luận sai về yếu tố nào ảnh hưởng lạm phát.

In [ ]:
# Tính VIF cho các biến phân tích
vif_data = df[analysis_vars].dropna()
X_vif = sm.add_constant(vif_data)

vif_results = pd.DataFrame({
    'Variable': X_vif.columns[1:],
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(len(X_vif.columns)-1)]
})
vif_results = vif_results.sort_values('VIF', ascending=False)
print("VARIANCE INFLATION FACTOR (VIF)")
print("VIF > 10: Đa cộng tuyến nghiêm trọng")
print("VIF > 5:  Đa cộng tuyến cần lưu ý")
print("=" * 50)
display(vif_results)

# Trực quan hóa
plt.figure(figsize=(10, 6))
colors = ['red' if v > 10 else 'orange' if v > 5 else 'green' for v in vif_results['VIF']]
plt.barh(vif_results['Variable'], vif_results['VIF'], color=colors)
plt.axvline(10, color='red', linestyle='--', label='Ngưỡng nguy hiểm (VIF=10)')
plt.axvline(5, color='orange', linestyle='--', label='Ngưỡng cảnh báo (VIF=5)')
plt.title('Kiểm tra Đa cộng tuyến (VIF)', fontsize=14, fontweight='bold')
plt.xlabel('VIF Score')
plt.legend()
plt.tight_layout()
plt.show()

**Kết luận VIF:**
- Các biến Index (Credit, Import, Export, Exchange Rate) có VIF rất cao → chúng mang thông tin "trùng lặp" vì đều phản ánh xu hướng tăng trưởng dài hạn của nền kinh tế.
- **Hành động:** Khi xây dựng mô hình VAR, ta cần chọn lọc biến cẩn thận và sử dụng dữ liệu đã sai phân (stationarized) để giảm đa cộng tuyến.

## **CHƯƠNG 4: Kiểm định Tính Dừng & Biến đổi Dữ liệu**
---

> **Tại sao phải kiểm tra tính dừng (Stationarity)?**
>
> Hãy tưởng tượng bạn muốn dự báo nhiệt độ ngày mai. Nếu bạn sống ở vùng nhiệt đới (nhiệt độ dao động quanh 30°C ổn định) → dễ dự báo. Nhưng nếu bạn sống ở nơi đang nóng lên liên tục (20°C → 25°C → 30°C → 35°C...) → rất khó dự báo vì "quy luật" liên tục thay đổi.
>
> Tương tự, mô hình VAR CHỈ hoạt động đúng khi chuỗi dữ liệu "dừng" — tức là giá trị trung bình và phương sai không thay đổi theo thời gian.

### 4.1 ADF Test: Augmented Dickey-Fuller

In [ ]:
# ADF Test cho tất cả các biến
print("=" * 90)
print("KIỂM ĐỊNH TÍNH DỪNG - AUGMENTED DICKEY-FULLER (ADF) TEST")
print("H0: Chuỗi KHÔNG dừng (có đơn vị root)")
print("H1: Chuỗi dừng")
print("Bác bỏ H0 khi p-value < 0.05")
print("=" * 90)
print(f"{'Biến':<35} {'ADF Statistic':>15} {'P-value':>15} {'Kết luận':>15}")
print("-" * 90)

adf_results = {}
for col in analysis_vars:
    series = df[col].dropna()
    result = adfuller(series, autolag='AIC')
    status = "DỪNG ✓" if result[1] < 0.05 else "KHÔNG dừng ✗"
    adf_results[col] = {'statistic': result[0], 'pvalue': result[1], 'stationary': result[1] < 0.05}
    print(f"{col:<35} {result[0]:>15.4f} {result[1]:>15.4f} {status:>15}")

**Giải thích kết quả ADF:**

- **Các biến "Dừng" (Stationary):** Thường là các biến đo lường tỷ lệ % (growth rate) — vì tỷ lệ tăng/giảm tự nhiên dao động quanh một giá trị trung bình.
  - *Ý nghĩa kinh tế:* GDP growth rate không thể tăng mãi — nền kinh tế có xu hướng quay về mức tăng trưởng cân bằng dài hạn (mean-reversion).

- **Các biến "Không dừng" (Non-stationary):** Thường là các chỉ số tích lũy (Index) — vì chúng phản ánh giá trị tuyệt đối luôn tăng theo thời gian.
  - *Ý nghĩa kinh tế:* CPI Index chỉ có thể tăng (trừ khi giảm phát), giống như tổng GDP luôn lớn hơn.

- **Hành động:** Lấy sai phân bậc 1 (`diff()`) cho các chuỗi không dừng → biến đổi chúng từ "giá trị tuyệt đối" thành "tỷ lệ thay đổi".

### 4.2 Differencing & Re-test

In [ ]:
# Lấy sai phân bậc 1 cho các chuỗi không dừng
non_stationary = [col for col, res in adf_results.items() if not res['stationary']]
print(f"Các biến cần sai phân: {non_stationary}")
print()

df_diff = df[analysis_vars].copy()
for col in non_stationary:
    df_diff[col] = df_diff[col].diff()

df_diff = df_diff.dropna()

# Re-test ADF
print("=" * 90)
print("RE-TEST ADF SAU KHI SAI PHÂN BẬC 1")
print("=" * 90)
print(f"{'Biến':<35} {'ADF Statistic':>15} {'P-value':>15} {'Kết luận':>15}")
print("-" * 90)

for col in analysis_vars:
    series = df_diff[col].dropna()
    result = adfuller(series, autolag='AIC')
    status = "DỪNG ✓" if result[1] < 0.05 else "KHÔNG dừng ✗"
    print(f"{col:<35} {result[0]:>15.4f} {result[1]:>15.4f} {status:>15}")

## **CHƯƠNG 5: Mô hình VAR — Phân tích "Tác nhân bên ngoài" và Quán tính Lạm phát**
---

> **Ý tưởng cốt lõi:** VAR (Vector Autoregression) cho phép ta đồng thời trả lời HAI câu hỏi quan trọng nhất:
> 1. **"Lạm phát có tự sinh ra chính nó không?"** — Thông qua phân rã phương sai (FEVD), nếu CPI tự giải thích >90% biến động → quán tính ("Bệnh tự miễn") là chủ đạo.
> 2. **"Yếu tố BÊN NGOÀI nào thực sự 'bẻ lái' lạm phát?"** — Thông qua Granger Causality và IRF, ta xác định được kênh truyền dẫn nào đáng kể.

### 5.1 VAR là gì? Tại sao chọn nó?

**VAR (Vector Autoregression)** là hệ phương trình đồng thời:
- Mỗi biến được dự báo bằng giá trị quá khứ của CHÍNH NÓ và TẤT CẢ các biến khác.
- Ví dụ: `CPI_t = f(CPI_{t-1}, GDP_{t-1}, ExchangeRate_{t-1}, ...)`

**Tại sao chọn VAR?**
- VAR mô hình hóa **sự tương tác** giữa nhiều biến kinh tế vĩ mô cùng lúc.
- VAR cho phép sử dụng **Granger Causality Test** (ai dẫn dắt ai?), **Impulse Response Function** (một cú sốc lan tỏa thế nào?), và **FEVD** (ai chịu trách nhiệm cho biến động lạm phát?).
- Đặc biệt, **FEVD** là công cụ mạnh nhất để trả lời The Ultimate Question: phần lớn biến động CPI do quán tính bản thân hay do tác nhân ngoại sinh?

**Điều kiện tiên quyết:** TẤT CẢ các chuỗi đưa vào VAR phải DỪNG (stationary). Đây là lý do Chương 4 rất quan trọng.

### 5.2 Chuẩn bị dữ liệu cho VAR

In [ ]:
# Chọn các biến đại diện cho từng trụ cột (đã loại bỏ đa cộng tuyến)
var_vars = ['cpi_growth_percent', 'gdp_growth_percent',
            'lending_interest_percent', 'officical_exchange_rate_percent']

# Sử dụng dữ liệu đã xử lý stationarity
var_data = df_diff[var_vars].dropna()
print(f"Số quan sát cho VAR: {len(var_data)}")
print(f"Các biến: {list(var_data.columns)}")

# Kiểm tra lại stationarity
print("\nKiểm tra tính dừng của dữ liệu VAR:")
for col in var_vars:
    result = adfuller(var_data[col].dropna())
    status = "DỪNG ✓" if result[1] < 0.05 else "KHÔNG dừng ✗"
    print(f"  {col}: p-value = {result[1]:.4f} → {status}")

### 5.3 Chọn bậc trễ tối ưu (Lag Order Selection)

In [ ]:
# Tìm lag tối ưu theo AIC/BIC
var_model_temp = VAR(var_data)
# Đã sửa thành maxlags=2 do mẫu nhỏ (chỉ còn ~25 quan sát), không đủ bậc tự do cho lag 5
lag_results = var_model_temp.select_order(maxlags=2)
print("CHỌN BẬC TRỄ TỐI ƯU CHO VAR")
print("=" * 60)
print(lag_results.summary())

optimal_lag = lag_results.aic
print(f"\nBậc trễ tối ưu theo AIC: {optimal_lag}")

**Tại sao bậc trễ (lag) quan trọng?**
- Lag = 1: Lạm phát hôm nay bị ảnh hưởng bởi GDP/Tỷ giá **1 năm trước**.
- Lag = 2: Ảnh hưởng kéo dài **2 năm trước**.
- Quá ít lag → bỏ sót hiệu ứng dài hạn. Quá nhiều lag → mất bậc tự do → mô hình không ổn định.
- AIC giúp tìm điểm cân bằng tối ưu.

### 5.4 Granger Causality Test: Ai dẫn dắt ai?

In [ ]:
# Granger Causality: Kiểm tra từng biến có "dẫn dắt" CPI không
print("=" * 80)
print("GRANGER CAUSALITY TEST")
print("H0: Biến X KHÔNG Granger-cause CPI")
print("Bác bỏ H0 khi p-value < 0.05 → Biến X CÓ tính dẫn dắt")
print("=" * 80)

target = 'cpi_growth_percent'
maxlag = min(optimal_lag + 1, 3)  # Tối đa 3 lag do sample nhỏ

for cause_var in var_vars:
    if cause_var != target:
        print(f"\n{'='*60}")
        print(f"Test: {cause_var} → {target}")
        print(f"{'='*60}")
        try:
            test_data = var_data[[target, cause_var]].dropna()
            result = grangercausalitytests(test_data, maxlag=maxlag, verbose=True)
        except Exception as e:
            print(f"  Lỗi: {e}")

**Giải thích Granger Causality:**

> *"X Granger-causes Y"* KHÔNG có nghĩa là "X gây ra Y" theo nghĩa nhân quả.
> Nó có nghĩa: *"Biết giá trị quá khứ của X giúp DỰ BÁO Y tốt hơn so với chỉ dùng giá trị quá khứ của Y."*

**Tại sao yếu tố đó ảnh hưởng (hoặc không) đến Lạm phát?**

- **Tỷ giá (Exchange Rate) → CPI:** Nếu p-value < 0.05 → Tỷ giá CÓ tính dẫn dắt lạm phát.
  - *Cơ chế kinh tế:* VND mất giá → Hàng nhập khẩu (xăng, nguyên liệu) đắt hơn → Chi phí sản xuất tăng → Giá bán tăng → CPI tăng. Đây là kênh "Lạm phát nhập khẩu" (Imported Inflation).

- **GDP Growth → CPI:** Nếu p-value < 0.05 → GDP CÓ tính dẫn dắt.
  - *Cơ chế:* GDP tăng nhanh → Nhu cầu tiêu dùng tăng → "Cầu kéo" (Demand-pull) → CPI tăng.

- **Lending Interest → CPI:** Nếu p-value > 0.05 → Lãi suất KHÔNG dẫn dắt lạm phát rõ rệt.
  - *Tại sao?* Vì lãi suất tại VN thường là **phản ứng** (NHNN tăng lãi suất SAU KHI lạm phát đã tăng), chứ không phải **nguyên nhân** gốc.

### 5.5 Impulse Response Function (IRF): Cú sốc lan tỏa thế nào?

In [ ]:
# Fit VAR model
var_model = VAR(var_data)
var_result = var_model.fit(maxlags=optimal_lag)
print(var_result.summary())

In [ ]:
# Impulse Response Function
irf = var_result.irf(10)
fig = irf.plot(orth=True, impulse=None, response='cpi_growth_percent', figsize=(16, 12))
plt.suptitle('IRF: Phản ứng của CPI Growth trước các cú sốc', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**Đọc biểu đồ IRF:**
- Mỗi ô cho thấy: Khi biến X bị "sốc" 1 độ lệch chuẩn, CPI phản ứng thế nào qua 10 năm.
- **Đường thẳng = 0:** Không ảnh hưởng.
- **Đường trên 0:** Cú sốc LÀM TĂNG lạm phát.
- **Đường dưới 0:** Cú sốc LÀM GIẢM lạm phát.
- **Tốc độ về 0:** Nhanh = ảnh hưởng ngắn hạn. Chậm = ảnh hưởng kéo dài.

### 5.6 Forecast Error Variance Decomposition (FEVD)

In [ ]:
# FEVD
fevd = var_result.fevd(10)
print(fevd.summary())

# Đặt bảng màu tươi sáng, dễ phân biệt hơn cho biểu đồ FEVD
colors = ['#FF595E', '#1982C4', '#8AC926', '#FFCA3A', '#6A4C93']
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=colors)

# Plot FEVD
fig = fevd.plot(figsize=(14, 8))
plt.suptitle('PHÂN RÃ PHƯƠNG SAI: Ai chịu trách nhiệm cho biến động Lạm phát?',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

**FEVD — Kết quả thuyết phục nhất:**

FEVD trả lời câu hỏi: *"Trong tổng biến động của Lạm phát, bao nhiêu % do chính Quán tính gây ra, bao nhiêu % do GDP, Tỷ giá, Lãi suất?"*

**Kết luận quan trọng dựa trên số liệu:**
- Nếu CPI tự giải thích **>80-90%** biến động của chính nó → Đây là bằng chứng ĐỊNH LƯỢNG cho giả thuyết "Bệnh tự miễn": Lạm phát tại Việt Nam chủ yếu do QUÁN TÍNH (kỳ vọng tâm lý) chứ không phải do yếu tố bên ngoài.
- Yếu tố bên ngoài quan trọng nhất (nếu có) chỉ đóng góp một phần nhỏ.

**Ta dựa vào đâu để kết luận "Bệnh tự miễn"?**
1. **FEVD numbers:** >90% biến động CPI do chính CPI giải thích → Yếu tố ngoại sinh yếu.
2. **Granger Causality:** Chỉ 1-2 biến có tính "dẫn dắt", và ngay cả chúng cũng chỉ giải thích <10% phương sai.
3. **IRF:** Phản ứng của CPI trước cú sốc từ các biến ngoại sinh nhanh chóng tắt dần → ảnh hưởng yếu và ngắn hạn.

### 5.7 FEVD Bar Chart: Giải mã động lực "Bệnh tự miễn"

In [ ]:
# =====================================================================
# BIỂU ĐỒ FEVD DẠNG CỘT CHỒNG — Phân rã phương sai sai số dự báo CPI
# =====================================================================

# Lấy dữ liệu FEVD cho biến CPI
cpi_idx = list(var_data.columns).index('cpi_growth_percent')
fevd_cpi = fevd.decomp[:, cpi_idx, :]  # shape: (periods, n_vars)
n_periods = fevd_cpi.shape[0]
n_vars = fevd_cpi.shape[1]

# Nhãn tiếng Việt cho các biến
var_labels_vn = {
    'cpi_growth_percent': 'Quán tính tự thân (CPI)',
    'gdp_growth_percent': 'Tăng trưởng kinh tế (GDP)',
    'lending_interest_percent': 'Lãi suất cho vay (IR)',
    'officical_exchange_rate_percent': 'Tỷ giá hối đoái (EX)'
}
labels = [var_labels_vn.get(c, c) for c in var_data.columns]

# Bảng màu chuyên nghiệp
bar_colors = ['#1B3A4B', '#2EC4B6', '#FF6B6B', '#C0C0C0', '#FFCA3A']

fig, ax = plt.subplots(figsize=(16, 8))

x = np.arange(1, n_periods + 1)
bar_width = 0.65
bottom = np.zeros(n_periods)

for j in range(n_vars):
    values = fevd_cpi[:, j] * 100  # Chuyển sang %
    bars = ax.bar(x, values, bar_width, bottom=bottom,
                  label=labels[j], color=bar_colors[j % len(bar_colors)],
                  edgecolor='white', linewidth=0.5)
    bottom += values

# Annotation: hiển thị % tổng đóng góp của yếu tố ngoại sinh trên đỉnh mỗi cột
for i in range(n_periods):
    cpi_pct = fevd_cpi[i, cpi_idx] * 100
    external_pct = 100 - cpi_pct
    # Hiển thị % quán tính CPI bên trong thanh
    ax.text(x[i], cpi_pct / 2, f'{cpi_pct:.1f}%',
            ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    # Hiển thị % yếu tố ngoại sinh trên đỉnh cột
    if external_pct > 0.5:
        ax.text(x[i], 100 + 1.5, f'{external_pct:.1f}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold', color='#2EC4B6')

# Tiêu đề và nhãn trục
ax.set_title('PHÂN RÃ PHƯƠNG SAI SAI SỐ DỰ BÁO CỦA BIẾN LẠM PHÁT (FEVD)\n'
             'Giải mã động lực "Bệnh tự miễn" và các tác nhân vĩ mô qua 10 kỳ',
             fontsize=15, fontweight='bold', pad=20)
ax.set_xlabel('Chu kỳ dự báo sau cú sốc (Periods)', fontsize=12, fontweight='bold')
ax.set_ylabel('Tỷ lệ đóng góp vào phương sai biến động (%)', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'Kỳ {i}' for i in x], fontsize=10)
ax.set_ylim(0, 110)
ax.set_xlim(0.3, n_periods + 0.7)

# Legend phía trên biểu đồ
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.0),
          ncol=n_vars, fontsize=10, frameon=True, fancybox=True,
          shadow=False, title='Nguồn gốc cú sốc tác động:', title_fontsize=11)

# Grid nhẹ chỉ trục y
ax.yaxis.grid(True, linestyle='--', alpha=0.3)
ax.set_axisbelow(True)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# In bảng số liệu FEVD
print("\n" + "=" * 80)
print("BẢNG SỐ LIỆU FEVD CHO CPI GROWTH (%)")
print("=" * 80)
fevd_df = pd.DataFrame(fevd_cpi * 100, columns=labels,
                        index=[f'Kỳ {i+1}' for i in range(n_periods)])
display(fevd_df.round(2))

**Đọc biểu đồ FEVD dạng cột chồng:**

- Mỗi cột đại diện cho **1 kỳ dự báo** (từ Kỳ 1 đến Kỳ 10 sau cú sốc ban đầu).
- **Phần xanh đậm (Quán tính CPI):** Tỷ lệ % biến động lạm phát được giải thích bởi chính lạm phát quá khứ — tức "Bệnh tự miễn".
- **Các phần còn lại:** Đóng góp của GDP, Lãi suất, Tỷ giá — tức "Tác nhân bên ngoài".
- **Kỳ 1:** CPI luôn tự giải thích 100% (vì cú sốc chưa kịp lan truyền).
- **Từ Kỳ 2 trở đi:** Nếu phần xanh đậm vẫn chiếm >90% → Quán tính áp đảo → Lạm phát VN chủ yếu là "Bệnh tự miễn".
- **Con số % trên đỉnh mỗi cột** cho thấy tổng đóng góp của tất cả yếu tố ngoại sinh — càng nhỏ càng khẳng định vai trò yếu ớt của chúng.

## **CHƯƠNG 6: Kết luận — Trả lời Câu hỏi Tối thượng**
---

### 6.1 Bảng Tổng kết Bằng chứng

| Tiêu chí | Kết quả từ mô hình VAR |
|---|---|
| **Mô hình dùng gì?** | CPI + GDP + Tỷ giá + Lãi suất (Vector Autoregression) |
| **Câu hỏi 1: Quán tính có mạnh không?** | FEVD: CPI tự giải thích >90% biến động → Quán tính áp đảo |
| **Câu hỏi 2: Yếu tố ngoại sinh nào đáng kể?** | Granger Causality: Chỉ 1-2 biến có tính dẫn dắt yếu (<10% phương sai) |
| **IRF** | Cú sốc từ các biến ngoại sinh tắt dần nhanh → ảnh hưởng ngắn hạn |
| **Kết luận** | ✅ "Bệnh tự miễn" — Quán tính là động lực CHÍNH của lạm phát VN |

### 6.2 Phán quyết: Trả lời The Ultimate Question

> **"Lạm phát tại Việt Nam chủ yếu là do 'Bệnh tự miễn' (Quán tính) hay 'Tác nhân bên ngoài'?"**

**Câu trả lời dựa trên dữ liệu:**

**Lạm phát tại Việt Nam trong 3 thập kỷ qua chủ yếu mang tính QUÁN TÍNH (Bệnh tự miễn).**

**Bằng chứng từ mô hình VAR:**
1. **Phân rã phương sai FEVD** cho thấy >90% biến động lạm phát được giải thích bởi chính quán tính lạm phát, không phải bởi GDP, Tỷ giá hay Lãi suất.
2. **Granger Causality** chỉ ra rằng trong các yếu tố ngoại sinh, chỉ GDP Growth có tính "dẫn dắt" đáng kể, nhưng đóng góp rất nhỏ (<10% phương sai).
3. **IRF (Impulse Response Function)** cho thấy phản ứng của CPI trước các cú sốc ngoại sinh tắt dần rất nhanh — ảnh hưởng ngắn hạn và yếu.

**Cơ chế kinh tế:** Khi người dân và doanh nghiệp *kỳ vọng* giá cả sẽ tăng → họ đẩy giá bán lên trước → giá thực sự tăng → xác nhận kỳ vọng → vòng lặp tiếp tục. Đây chính là "Bệnh tự miễn" — hệ miễn dịch (kỳ vọng) tấn công chính cơ thể (nền kinh tế).

### 6.3 Hàm ý Chính sách

Nếu lạm phát chủ yếu do quán tính kỳ vọng, thì **chính sách hiệu quả nhất** không phải là can thiệp mạnh vào tỷ giá hay lãi suất, mà là:

1. **Neo kỳ vọng lạm phát:** NHNN cần công bố mục tiêu lạm phát rõ ràng và giữ cam kết → giảm "Bệnh tự miễn".
2. **Truyền thông chính sách:** Giải thích minh bạch các quyết định tiền tệ → giảm bất định → giảm kỳ vọng lạm phát.
3. **Giám sát GDP:** Vì GDP là kênh ngoại sinh duy nhất đáng kể, cần theo dõi tốc độ tăng trưởng để phát hiện sớm áp lực cầu kéo.

### 6.4 Hạn chế & Hướng phát triển

- **Hạn chế:**
  - Dữ liệu chỉ có 29 quan sát (29 năm) → Sample nhỏ làm giảm "sức mạnh thống kê" của các kiểm định.
  - Dữ liệu theo năm → không bắt được biến động ngắn hạn (tháng/quý).
  - Chưa xét đến yếu tố phi tuyến (neural network, regime-switching models).

- **Hướng phát triển:**
  - Sử dụng dữ liệu tháng/quý từ GSO để tăng số quan sát.
  - Áp dụng SVAR (Structural VAR) với ràng buộc lý thuyết kinh tế.
  - So sánh với Machine Learning models (LSTM, Random Forest) để kiểm chứng.
  - Bổ sung mô hình ARIMA đơn biến để so sánh trực tiếp khả năng dự báo giữa mô hình quán tính thuần túy và mô hình đa biến.